# 08 CNN Random Search

Joint HPO using the screened domain. Exactly 40 unique configurations, validation macro-F1 objective, no test use.


## 1. Package Setup


In [1]:
# Purpose: Package installs are documented but not run automatically during this static refactor.
# %pip install tensorflow scikit-learn pandas numpy matplotlib seaborn joblib soundfile librosa


## 2. Load Cache and Search Space


In [2]:
import os
from pathlib import Path

# Purpose: Mounts Google Drive when this notebook is running in Google Colab.
# Why this exists: the project files, cached MFCC features, manifests, models, figures,
# and metric outputs live in Google Drive during Colab runs. The /content/drive path
# only represents the real MyDrive files after drive.mount("/content/drive") succeeds.
# Important: the project root should be the folder that contains Data, Model Variants,
# and outputs. For this project, that expected Colab folder is the INM701 folder below.
COLAB_DRIVE_MOUNT_POINT = Path("/content/drive")
EXPECTED_COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")

try:
    from google.colab import drive

    drive.mount(str(COLAB_DRIVE_MOUNT_POINT))
    print("Google Colab detected. Google Drive mounted.")

    current_project_root = os.environ.get("INTRO_AI_PROJECT_ROOT")
    current_data_dir_exists = bool(current_project_root) and (Path(current_project_root) / "Data").exists()
    expected_data_dir_exists = (EXPECTED_COLAB_PROJECT_ROOT / "Data").exists()

    # Purpose: Keep a valid user-provided project root, but repair stale runtime state
    # if a previous cell pointed INTRO_AI_PROJECT_ROOT somewhere that does not contain Data.
    if current_data_dir_exists:
        print("INTRO_AI_PROJECT_ROOT already points to a folder with Data, so it was kept.")
    elif expected_data_dir_exists:
        os.environ["INTRO_AI_PROJECT_ROOT"] = str(EXPECTED_COLAB_PROJECT_ROOT)
        print("INTRO_AI_PROJECT_ROOT set to the expected INM701 project folder.")
    elif not current_project_root:
        os.environ["INTRO_AI_PROJECT_ROOT"] = str(EXPECTED_COLAB_PROJECT_ROOT)
        print("INTRO_AI_PROJECT_ROOT was not set, so it now points to the expected INM701 folder.")
    else:
        print("INTRO_AI_PROJECT_ROOT was kept, but Data was not found there or in the expected INM701 folder.")
except Exception as exc:
    print("Google Colab Drive mount skipped. This is expected outside Colab.")
    print("Mount skip reason:", exc)

active_project_root = os.environ.get("INTRO_AI_PROJECT_ROOT", "not set")
print("INTRO_AI_PROJECT_ROOT:", active_project_root)
if active_project_root != "not set":
    active_project_root = Path(active_project_root)
    print("Project root exists:", active_project_root.exists())
    print("Expected Data folder:", active_project_root / "Data")
    print("Data folder exists:", (active_project_root / "Data").exists())


Mounted at /content/drive
Google Colab detected. Google Drive mounted.
INTRO_AI_PROJECT_ROOT set to the expected INM701 project folder.
INTRO_AI_PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Education/INM701
Project root exists: True
Expected Data folder: /content/drive/MyDrive/Colab Notebooks/Education/INM701/Data
Data folder exists: True


In [3]:
# Purpose: Later notebooks load the one CNN-ready cache made by notebook 00. They do not
# rescan folders, regenerate splits, extract MFCCs, fit scalers, or recalculate class weights.
import hashlib
import json
import os
import random
import time
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Conv2D, Dense, Dropout, GlobalAveragePooling2D, Input, MaxPooling2D
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 7
EARLY_STOP_MIN_DELTA = 1e-4
THRESHOLD = 0.5
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}
CONFIRMATION_SEEDS = [42, 123, 2026]


def resolve_project_root():
    # Purpose: default path.
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    # Purpose: Runs each candidate configuration under the same data split for a fair validation comparison.
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


# Purpose: Centralizes filesystem paths so dataset inputs, caches, figures, models, and metric tables are easy
# Purpose: to trace.
PROJECT_ROOT = resolve_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cnn"
CACHE_DIR = OUTPUT_DIR / "cache"
MANIFESTS_DIR = OUTPUT_DIR / "manifests"
CONFIGS_DIR = OUTPUT_DIR / "configs"
TABLES_DIR = OUTPUT_DIR / "tables"
HISTORIES_DIR = OUTPUT_DIR / "histories"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
PILOT_LEGACY_DIR = OUTPUT_DIR / "pilot_legacy"
# Purpose: Creates each output directory before later cells try to save tables, figures, or models.
for directory in [OUTPUT_DIR, CACHE_DIR, MANIFESTS_DIR, CONFIGS_DIR, TABLES_DIR, HISTORIES_DIR, METRICS_DIR, FIGURES_DIR, MODELS_DIR, PILOT_LEGACY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    # Purpose: Stops the notebook early with a clear message when a required upstream artifact is missing.
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")
    return path


def load_json(path):
    # Purpose: Keeps the load_json helper isolated so later notebook cells can call it consistently.
    with open(require_file(path), "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(payload, path):
    # Purpose: Keeps the save_json helper isolated so later notebook cells can call it consistently.
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def load_class_weights():
    # Purpose: Keeps the load_class_weights helper isolated so later notebook cells can call it consistently.
    payload = load_json(CONFIGS_DIR / "class_weights.json")
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


for required in [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    CONFIGS_DIR / "class_weights.json",
]:
    require_file(required)

X_train = np.load(CACHE_DIR / "X_train.npy", mmap_mode="r")[..., np.newaxis]
y_train = np.load(CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation = np.load(CACHE_DIR / "X_validation.npy", mmap_mode="r")[..., np.newaxis]
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(CACHE_DIR / "validation_metadata.csv")
feature_config = load_json(CACHE_DIR / "feature_config.json")
CLASS_WEIGHTS = load_class_weights()

if len(X_train) != len(y_train) or len(X_train) != len(train_metadata):
    raise RuntimeError("Training cache arrays, labels and metadata are not row-aligned.")
if len(X_validation) != len(y_validation) or len(X_validation) != len(validation_metadata):
    raise RuntimeError("Validation cache arrays, labels and metadata are not row-aligned.")

print("Loaded CNN cache:", CACHE_DIR)
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)

search_payload = load_json(CONFIGS_DIR / "screened_hpo_search_space.json")
SEARCH_SPACE = search_payload["search_space"]


Loaded CNN cache: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/cnn/cache
X_train shape: (9060, 40, 502, 1)
X_validation shape: (1942, 40, 502, 1)


## 3. Keras Utilities


In [4]:
# Purpose: Shared CNN utilities for validation-only model selection.
def normalise_config(config):
    # Purpose: Converts JSON/CSV values into the types expected by the CNN builder.
    normalised = dict(config)
    normalised["conv_filters"] = [int(value) for value in normalised["conv_filters"]]
    normalised["kernel_size"] = [int(value) for value in normalised["kernel_size"]]
    normalised["dense_units"] = int(normalised["dense_units"])
    normalised["activation"] = str(normalised.get("activation", "relu"))
    normalised["optimizer"] = str(normalised.get("optimizer", "adam"))
    normalised["dropout"] = float(normalised["dropout"])
    normalised["learning_rate"] = float(normalised["learning_rate"])
    normalised["batch_size"] = int(normalised["batch_size"])
    normalised["loss"] = normalised.get("loss", "binary_crossentropy")
    normalised["threshold"] = float(normalised.get("threshold", THRESHOLD))
    return normalised


def config_json(config):
    # Purpose: Creates a stable representation for deduplication and deterministic seeds.
    return json.dumps(normalise_config(config), sort_keys=True)


def config_key(config):
    return hashlib.sha256(config_json(config).encode("utf-8")).hexdigest()


def seed_from_config(config, base_seed=RANDOM_STATE):
    digest = hashlib.sha256(f"{base_seed}:{config_json(config)}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % (2**31 - 1)


def set_global_seed(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    tf.keras.utils.set_random_seed(int(seed))
    os.environ["PYTHONHASHSEED"] = str(int(seed))


def make_optimizer(config):
    name = str(config["optimizer"]).lower()
    learning_rate = float(config["learning_rate"])
    if name == "adam":
        return tf.keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "rmsprop":
        return tf.keras.optimizers.RMSprop(learning_rate=learning_rate)
    if name in ["sgd", "sgd_momentum"]:
        return tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    raise ValueError(f"Unsupported optimizer: {config['optimizer']}")


def build_cnn_model(config, input_shape):
    # Purpose: Learns local patterns from the coefficient-by-time MFCC map.
    config = normalise_config(config)
    model = Sequential(name="mfcc_cnn")
    model.add(Input(shape=tuple(input_shape)))
    for filters in config["conv_filters"]:
        model.add(
            Conv2D(
                filters=filters,
                kernel_size=tuple(config["kernel_size"]),
                padding="same",
                strides=(1, 1),
                activation=config["activation"],
            )
        )
        model.add(MaxPooling2D(pool_size=(2, 2)))
        if config["dropout"] > 0:
            model.add(Dropout(config["dropout"]))
    # Global pooling controls parameter growth when the number of MFCC time frames changes.
    model.add(GlobalAveragePooling2D())
    model.add(Dense(config["dense_units"], activation=config["activation"]))
    if config["dropout"] > 0:
        model.add(Dropout(config["dropout"]))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(loss=config["loss"], optimizer=make_optimizer(config), metrics=["accuracy"])
    return model


def train_and_evaluate_config(config, run_seed, run_name, verbose=0):
    # Purpose: Uses training data for fitting and validation data for selection; never test data.
    config = normalise_config(config)
    tf.keras.backend.clear_session()
    set_global_seed(run_seed)
    model = build_cnn_model(config, X_train.shape[1:])
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            min_delta=EARLY_STOP_MIN_DELTA,
            restore_best_weights=True,
        )
    ]
    start = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=MAX_EPOCHS,
        batch_size=config["batch_size"],
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks,
        verbose=verbose,
    )
    runtime = time.perf_counter() - start
    probability = model.predict(X_validation, batch_size=config["batch_size"], verbose=0).ravel()
    pred = (probability >= config["threshold"]).astype(int)
    row = {
        "configuration": config,
        "config_json": config_json(config),
        "config_key": config_key(config),
        "seed": int(run_seed),
        "validation_macro_f1": float(f1_score(y_validation, pred, average="macro", zero_division=0)),
        "validation_binary_f1_synthetic": float(f1_score(y_validation, pred, pos_label=1, zero_division=0)),
        "validation_accuracy": float(accuracy_score(y_validation, pred)),
        "validation_precision_synthetic": float(precision_score(y_validation, pred, pos_label=1, zero_division=0)),
        "validation_recall_synthetic": float(recall_score(y_validation, pred, pos_label=1, zero_division=0)),
        "best_validation_loss": float(np.min(history.history["val_loss"])),
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "epochs_trained": int(len(history.history["loss"])),
        "runtime_seconds": float(runtime),
        "run_name": run_name,
    }
    history_df = pd.DataFrame(history.history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
    return row, history_df


def flat_result(row, extra=None):
    # Purpose: Makes each CNN configuration readable in CSV result tables.
    extra = extra or {}
    config = normalise_config(row["configuration"])
    flat = dict(extra)
    flat.update({
        "conv_filters": json.dumps(config["conv_filters"]),
        "kernel_size": json.dumps(config["kernel_size"]),
        "dense_units": config["dense_units"],
        "activation": config["activation"],
        "optimizer": config["optimizer"],
        "dropout": config["dropout"],
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "loss": config["loss"],
        "threshold": config["threshold"],
        "config_json": row["config_json"],
        "config_key": row["config_key"],
        "seed": row["seed"],
        "validation_macro_f1": row["validation_macro_f1"],
        "validation_binary_f1_synthetic": row["validation_binary_f1_synthetic"],
        "validation_accuracy": row["validation_accuracy"],
        "validation_precision_synthetic": row["validation_precision_synthetic"],
        "validation_recall_synthetic": row["validation_recall_synthetic"],
        "best_validation_loss": row["best_validation_loss"],
        "best_epoch": row["best_epoch"],
        "epochs_trained": row["epochs_trained"],
        "runtime_seconds": row["runtime_seconds"],
    })
    return flat


def select_best(rows):
    return max(rows, key=lambda row: (row["validation_macro_f1"], -row["best_validation_loss"]))


## 4. Run or Load Random Search


In [5]:
# Purpose: Runs or reloads Random Search; rebuilds the best-config JSON from trial CSVs when needed.
RUN_RANDOM_SEARCH = True
RANDOM_SEARCH_EVALUATIONS = 40
trials_path = TABLES_DIR / "08_random_search_trials.csv"
best_path = CONFIGS_DIR / "08_random_search_best.json"

def rebuild_best_json_from_trials(trials_path, best_path, method_name):
    # Purpose: Rebuilds a missing best-config JSON from an existing search trial CSV.
    # This keeps the HPO handoff usable without rerunning expensive model training.
    trials_path = Path(trials_path)
    best_path = Path(best_path)
    if not trials_path.exists():
        print("Cannot rebuild HPO best config because trial CSV is missing:", trials_path)
        return None

    trials_df = pd.read_csv(trials_path)
    required_columns = {"config_json", "validation_macro_f1", "best_validation_loss"}
    missing_columns = required_columns - set(trials_df.columns)
    if missing_columns:
        raise RuntimeError(f"Cannot rebuild {best_path.name}; missing columns in {trials_path.name}: {sorted(missing_columns)}")
    if trials_df.empty:
        raise RuntimeError(f"Cannot rebuild {best_path.name}; trial CSV is empty: {trials_path}")

    unique_df = trials_df.drop_duplicates("config_key") if "config_key" in trials_df.columns else trials_df
    best_row = unique_df.sort_values(["validation_macro_f1", "best_validation_loss"], ascending=[False, True]).iloc[0]
    payload = best_row.to_dict()
    payload["method"] = method_name
    payload["configuration"] = normalise_config(json.loads(best_row["config_json"]))
    payload["selection_metric"] = "validation_macro_f1"
    payload["rebuilt_from_trials_csv"] = str(trials_path)
    save_json(payload, best_path)
    print("Rebuilt HPO best config from existing trials:", best_path)
    return payload

def enumerate_space(space):
    # Purpose: Expands the screened HPO search space into concrete configurations before
    # Random Search samples a fixed number of unique candidates.
    keys = list(space)
    return [normalise_config(dict(zip(keys, values))) for values in product(*(space[key] for key in keys))]

all_configs = enumerate_space(SEARCH_SPACE)
if len(all_configs) < RANDOM_SEARCH_EVALUATIONS:
    raise RuntimeError("Random Search needs at least 40 unique configurations.")

if RUN_RANDOM_SEARCH:
    rng = np.random.default_rng(RANDOM_STATE)
    indices = rng.choice(len(all_configs), size=RANDOM_SEARCH_EVALUATIONS, replace=False)
    rows = []
    # Purpose: Trains each sampled HPO configuration once and appends the result to the
    # trial CSV after every completed evaluation.
    for number, index in enumerate(indices, start=1):
        config = all_configs[int(index)]
        row, _ = train_and_evaluate_config(config, seed_from_config(config), f"08_random_{number:03d}", verbose=2)
        row.update({"method": "random_search", "unique_evaluation_number": number})
        rows.append(row)
        pd.DataFrame([flat_result(item, {"method": "random_search", "unique_evaluation_number": item["unique_evaluation_number"]}) for item in rows]).to_csv(trials_path, index=False)
    best = select_best(rows)
    save_json(flat_result(best, {"method": "random_search"}), best_path)
elif best_path.exists():
    best_payload = load_json(best_path)
    print("Loaded HPO best config:", best_path)
    if trials_path.exists():
        display(pd.read_csv(trials_path).sort_values(["validation_macro_f1", "best_validation_loss"], ascending=[False, True]).head())
    else:
        display(pd.DataFrame([best_payload]))
elif trials_path.exists():
    rebuild_best_json_from_trials(trials_path, best_path, "random_search")
    display(pd.read_csv(trials_path).sort_values(["validation_macro_f1", "best_validation_loss"], ascending=[False, True]).head())
else:
    print("RUN_RANDOM_SEARCH is False. [RESULT TO BE INSERTED AFTER FINAL RUN]")
    print("Missing best config:", best_path)
    print("Missing trial CSV:", trials_path)


Epoch 1/50
284/284 - 24s - 85ms/step - accuracy: 0.6184 - loss: 0.6320 - val_accuracy: 0.6607 - val_loss: 0.6170
Epoch 2/50
284/284 - 2s - 7ms/step - accuracy: 0.7670 - loss: 0.4585 - val_accuracy: 0.8507 - val_loss: 0.3486
Epoch 3/50
284/284 - 2s - 7ms/step - accuracy: 0.8256 - loss: 0.3640 - val_accuracy: 0.8661 - val_loss: 0.3095
Epoch 4/50
284/284 - 2s - 7ms/step - accuracy: 0.8535 - loss: 0.3147 - val_accuracy: 0.8702 - val_loss: 0.2948
Epoch 5/50
284/284 - 2s - 7ms/step - accuracy: 0.8858 - loss: 0.2567 - val_accuracy: 0.8800 - val_loss: 0.2671
Epoch 6/50
284/284 - 2s - 7ms/step - accuracy: 0.9093 - loss: 0.2115 - val_accuracy: 0.9212 - val_loss: 0.1935
Epoch 7/50
284/284 - 2s - 7ms/step - accuracy: 0.9223 - loss: 0.1865 - val_accuracy: 0.9336 - val_loss: 0.1725
Epoch 8/50
284/284 - 2s - 7ms/step - accuracy: 0.9253 - loss: 0.1848 - val_accuracy: 0.9464 - val_loss: 0.1411
Epoch 9/50
284/284 - 2s - 7ms/step - accuracy: 0.9342 - loss: 0.1638 - val_accuracy: 0.9310 - val_loss: 0.1785

## 5. Checks


In [6]:
# Purpose: Runs this notebook step and prints or saves the resulting intermediate output for review.
display(pd.DataFrame([("unique_budget", 40), ("same_domain_as_ga", True), ("seed_rule", "sha256_config_specific"), ("test_metrics_computed", False)], columns=["check", "value"]))


,check,value
0,unique_budget,40
1,same_domain_as_ga,True
2,seed_rule,sha256_config_specific
3,test_metrics_computed,False
